# Insurance Pipeline – Advanced Data Engineering Assignment

**Domain:** Insurance | **Stack:** PySpark + Spark SQL

> Pipeline: Customer → Policy → Claim → Agent
> Goal: Maximize premium insight, minimize risk exposure

## Phase 1 – Data Understanding

#### Load all tables from CSV

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

BASE = "/Volumes/workspace/default/week2-day1"

customers    = spark.read.csv(f"{BASE}/customers.csv",    header=True, inferSchema=True)
policies     = spark.read.csv(f"{BASE}/policies.csv",     header=True, inferSchema=True)
claims       = spark.read.csv(f"{BASE}/claims.csv",       header=True, inferSchema=True)
agents       = spark.read.csv(f"{BASE}/agents.csv",       header=True, inferSchema=True)
policy_agent = spark.read.csv(f"{BASE}/policy_agent.csv", header=True, inferSchema=True)

print("All tables loaded.")


All tables loaded.


#### Schema inspection

In [0]:
for label, df in [("customers", customers), ("policies", policies),
                  ("claims", claims), ("agents", agents), ("policy_agent", policy_agent)]:
    print(f"\n{'='*50}")
    print(f"Table: {label}")
    df.printSchema()



Table: customers
root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- city: string (nullable = true)


Table: policies
root
 |-- policy_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- policy_type: string (nullable = true)
 |-- premium: integer (nullable = true)
 |-- start_date: date (nullable = true)


Table: claims
root
 |-- claim_id: integer (nullable = true)
 |-- policy_id: integer (nullable = true)
 |-- claim_amount: integer (nullable = true)
 |-- claim_date: date (nullable = true)
 |-- status: string (nullable = true)


Table: agents
root
 |-- agent_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- region: string (nullable = true)


Table: policy_agent
root
 |-- policy_id: integer (nullable = true)
 |-- agent_id: integer (nullable = true)



#### Row counts

In [0]:
for label, df in [("customers", customers), ("policies", policies),
                  ("claims", claims), ("agents", agents), ("policy_agent", policy_agent)]:
    print(f"{label:15s}: {df.count()} rows")


customers      : 60 rows
policies       : 80 rows
claims         : 80 rows
agents         : 20 rows
policy_agent   : 80 rows


#### Null value check per table

In [0]:
for label, df in [("customers", customers), ("policies", policies),
                  ("claims", claims), ("agents", agents), ("policy_agent", policy_agent)]:
    print(f"\n--- {label} ---")
    df.select([sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()



--- customers ---
+-----------+----+---+----+
|customer_id|name|age|city|
+-----------+----+---+----+
|          0|   0|  0|   0|
+-----------+----+---+----+


--- policies ---
+---------+-----------+-----------+-------+----------+
|policy_id|customer_id|policy_type|premium|start_date|
+---------+-----------+-----------+-------+----------+
|        0|          0|          0|      0|         0|
+---------+-----------+-----------+-------+----------+


--- claims ---
+--------+---------+------------+----------+------+
|claim_id|policy_id|claim_amount|claim_date|status|
+--------+---------+------------+----------+------+
|       0|        0|           0|         0|     0|
+--------+---------+------------+----------+------+


--- agents ---
+--------+----+------+
|agent_id|name|region|
+--------+----+------+
|       0|   0|     0|
+--------+----+------+


--- policy_agent ---
+---------+--------+
|policy_id|agent_id|
+---------+--------+
|        0|       0|
+---------+--------+



#### Identify negative values

In [0]:
print("Policies with negative premium:")
policies.filter(col("premium") < 0).show()

print("Claims with negative claim_amount:")
claims.filter(col("claim_amount") < 0).show()


Policies with negative premium:
+---------+-----------+-----------+-------+----------+
|policy_id|customer_id|policy_type|premium|start_date|
+---------+-----------+-----------+-------+----------+
|      103|         32|       Life|  -1484|2023-06-29|
|      105|         50|       Life|  -4031|2023-08-31|
|      106|         53|     Health|  -3312|2023-08-14|
|      107|         33|     Health|  -4094|2023-03-02|
|      114|         23|       Auto|  -1102|2023-10-08|
|      115|         16|       Auto|  -3460|2023-03-11|
|      124|          5|       Life|  -3927|2023-05-12|
|      126|         10|     Health|  -2972|2023-06-27|
|      129|         52|     Health|   -993|2023-06-15|
|      132|          3|     Health|  -4360|2023-07-28|
|      133|         65|       Life|  -2649|2023-09-07|
|      134|         28|       Auto|   -733|2023-10-08|
|      139|         37|       Auto|  -2390|2023-05-15|
|      140|         61|       Auto|   -830|2023-04-14|
|      141|          5|       Aut

#### Identify duplicate rows

In [0]:
for label, df in [("customers", customers), ("policies", policies), ("claims", claims)]:
    dupes = df.count() - df.distinct().count()
    print(f"{label}: {dupes} duplicate rows")


customers: 0 duplicate rows
policies: 0 duplicate rows
claims: 0 duplicate rows


## Phase 2 – Data Cleaning

#### Fix negative premium → absolute value; correct data types

In [0]:
policies_clean = policies     .withColumn("premium",    abs(col("premium")))     .withColumn("start_date", to_date(col("start_date")))

print("Negative premiums remaining:", policies_clean.filter(col("premium") < 0).count())
policies_clean.show(5)


Negative premiums remaining: 0
+---------+-----------+-----------+-------+----------+
|policy_id|customer_id|policy_type|premium|start_date|
+---------+-----------+-----------+-------+----------+
|      101|         46|       Life|    378|2023-08-12|
|      102|         60|       Auto|  13450|2023-09-10|
|      103|         32|       Life|   1484|2023-06-29|
|      104|         53|       Auto|  16546|2023-08-19|
|      105|         50|       Life|   4031|2023-08-31|
+---------+-----------+-----------+-------+----------+
only showing top 5 rows


#### Fix negative claim_amount → absolute value; correct data types

In [0]:
claims_clean = claims     .withColumn("claim_amount", abs(col("claim_amount")))     .withColumn("claim_date",   to_date(col("claim_date")))

print("Negative claim_amounts remaining:", claims_clean.filter(col("claim_amount") < 0).count())
claims_clean.show(5)


Negative claim_amounts remaining: 0
+--------+---------+------------+----------+--------+
|claim_id|policy_id|claim_amount|claim_date|  status|
+--------+---------+------------+----------+--------+
|     201|      128|        2016|2023-07-04|Rejected|
|     202|      189|        3524|2023-06-10|Approved|
|     203|      151|        6667|2023-02-05|Rejected|
|     204|      170|        5569|2023-02-15|Rejected|
|     205|      153|       13249|2023-09-01|Rejected|
+--------+---------+------------+----------+--------+
only showing top 5 rows


#### Trim and standardize string columns

In [0]:
customers_clean = customers     .withColumn("name", trim(col("name")))     .withColumn("city", trim(col("city")))

agents_clean = agents     .withColumn("name",   trim(col("name")))     .withColumn("region", trim(col("region")))

policies_clean = policies_clean     .withColumn("policy_type", trim(col("policy_type")))

print("String columns trimmed.")


String columns trimmed.


#### Drop rows with null critical keys

In [0]:
before = {"policies": policies_clean.count(), "claims": claims_clean.count()}

policies_clean = policies_clean.dropna(subset=["policy_id", "customer_id"])
claims_clean   = claims_clean.dropna(subset=["claim_id", "policy_id"])

for t, df in [("policies", policies_clean), ("claims", claims_clean)]:
    print(f"{t}: {before[t]} → {df.count()} rows after dropping null critical keys")


policies: 80 → 80 rows after dropping null critical keys
claims: 80 → 80 rows after dropping null critical keys


## Phase 3 – Data Validation

#### Left anti joins — find invalid foreign keys

In [0]:
# Policies with no matching customer
invalid_policies = policies_clean.join(customers_clean, "customer_id", "left_anti")
print(f"Policies with invalid customer_id : {invalid_policies.count()}")
invalid_policies.show()

# Claims with no matching policy
invalid_claims = claims_clean.join(policies_clean, "policy_id", "left_anti")
print(f"Claims with invalid policy_id     : {invalid_claims.count()}")
invalid_claims.show()

# policy_agent with no matching policy
invalid_pa = policy_agent.join(policies_clean, "policy_id", "left_anti")
print(f"policy_agent rows with invalid policy_id: {invalid_pa.count()}")
invalid_pa.show()


Policies with invalid customer_id : 10
+-----------+---------+-----------+-------+----------+
|customer_id|policy_id|policy_type|premium|start_date|
+-----------+---------+-----------+-------+----------+
|         70|      118|       Auto|   1960|2023-08-21|
|         69|      120|       Life|   2812|2023-01-09|
|         65|      122|       Auto|   3456|2023-05-07|
|         63|      125|     Health|  14463|2023-02-17|
|         65|      133|       Life|   2649|2023-09-07|
|         61|      140|       Auto|    830|2023-04-14|
|         69|      144|       Auto|   4821|2023-10-13|
|         67|      148|     Health|   8126|2023-02-20|
|         65|      159|     Health|  14550|2023-06-02|
|         61|      162|     Health|    461|2023-08-30|
+-----------+---------+-----------+-------+----------+

Claims with invalid policy_id     : 13
+---------+--------+------------+----------+--------+
|policy_id|claim_id|claim_amount|claim_date|  status|
+---------+--------+------------+----------

#### Remove invalid foreign key rows

In [0]:
policies_valid = policies_clean.join(customers_clean.select("customer_id"), "customer_id", "inner")
claims_valid   = claims_clean.join(policies_valid.select("policy_id"), "policy_id", "inner")

print(f"policies after FK cleanup : {policies_valid.count()}")
print(f"claims   after FK cleanup : {claims_valid.count()}")


policies after FK cleanup : 70
claims   after FK cleanup : 61


#### Validation report

In [0]:
print("=" * 50)
print("VALIDATION REPORT")
print("=" * 50)
print(f"  customers          : {customers_clean.count()} rows (clean)")
print(f"  policies raw       : {policies.count()} | cleaned: {policies_valid.count()}")
print(f"  claims   raw       : {claims.count()}   | cleaned: {claims_valid.count()}")
print(f"  agents             : {agents_clean.count()} rows (clean)")
print(f"  policy_agent       : {policy_agent.count()} rows")
print(f"  Invalid policies   : {invalid_policies.count()}")
print(f"  Invalid claims     : {invalid_claims.count()}")
print(f"  Negative premiums fixed  : {policies.filter(col('premium') < 0).count()}")
print(f"  Negative claims fixed    : {claims.filter(col('claim_amount') < 0).count()}")
print("=" * 50)


VALIDATION REPORT
  customers          : 60 rows (clean)
  policies raw       : 80 | cleaned: 70
  claims   raw       : 80   | cleaned: 61
  agents             : 20 rows (clean)
  policy_agent       : 80 rows
  Invalid policies   : 10
  Invalid claims     : 13
  Negative premiums fixed  : 24
  Negative claims fixed    : 10


## Phase 4 – Transformations

#### Build clean joined dataframe (no duplication)

In [0]:
# Aggregate BEFORE joining to avoid fan-out duplication
cust_premium = policies_valid     .groupBy("customer_id")     .agg(
        sum("premium").alias("total_premium"),
        count("policy_id").alias("policy_count")
    )

cust_claims = claims_valid     .join(policies_valid.select("policy_id", "customer_id"), "policy_id")     .groupBy("customer_id")     .agg(
        sum("claim_amount").alias("total_claim"),
        count("claim_id").alias("claim_count")
    )

print("Premiums per customer (sample):")
cust_premium.show(5)
print("Claims per customer (sample):")
cust_claims.show(5)


Premiums per customer (sample):
+-----------+-------------+------------+
|customer_id|total_premium|policy_count|
+-----------+-------------+------------+
|         12|        25754|           2|
|         18|        39181|           3|
|         16|        14656|           2|
|          5|         5836|           2|
|         10|         2972|           1|
+-----------+-------------+------------+
only showing top 5 rows
Claims per customer (sample):
+-----------+-----------+-----------+
|customer_id|total_claim|claim_count|
+-----------+-----------+-----------+
|         18|      19870|          5|
|         12|       6667|          1|
|         16|      24500|          2|
|         10|       3636|          1|
|         31|      12294|          1|
+-----------+-----------+-----------+
only showing top 5 rows


#### Compute risk score per customer

In [0]:
customer_metrics = customers_clean     .join(cust_premium, "customer_id", "left")     .join(cust_claims,  "customer_id", "left")     .fillna(0, subset=["total_premium", "total_claim", "policy_count", "claim_count"])     .withColumn("risk_score",
        when(col("total_premium") == 0, lit(None))
        .otherwise(round(col("total_claim") / col("total_premium"), 4))
    )

print("Customer metrics (top 10 by risk score):")
customer_metrics.orderBy(col("risk_score").desc()).display(10)


Customer metrics (top 10 by risk score):


customer_id,name,age,city,total_premium,policy_count,total_claim,claim_count,risk_score
46,Cust46,31,Hyderabad,378,1,7373,1,19.5053
23,Cust23,30,Mumbai,1102,1,13117,1,11.9029
54,Cust54,25,Chennai,2962,1,23605,2,7.9693
58,Cust58,29,Bangalore,2618,1,10908,1,4.1665
55,Cust55,27,Mumbai,1322,1,5394,1,4.0802
6,Cust6,50,Mumbai,3629,1,12245,1,3.3742
21,Cust21,45,Chennai,6560,2,19279,3,2.9389
5,Cust5,36,Hyderabad,5836,2,14832,3,2.5415
2,Cust2,52,Chennai,872,1,2059,1,2.3612
19,Cust19,35,Hyderabad,17219,2,36814,5,2.138


#### City-wise premium and claim distribution

In [0]:
city_distribution = customer_metrics     .groupBy("city")     .agg(
        sum("total_premium").alias("city_total_premium"),
        sum("total_claim").alias("city_total_claim"),
        count("customer_id").alias("customer_count"),
        round(avg("risk_score"), 4).alias("avg_risk_score")
    )     .orderBy(col("city_total_premium").desc())

print("City-wise distribution:")
city_distribution.show()


City-wise distribution:
+---------+------------------+----------------+--------------+--------------+
|     city|city_total_premium|city_total_claim|customer_count|avg_risk_score|
+---------+------------------+----------------+--------------+--------------+
|Bangalore|            175610|          131478|            15|        1.0485|
|   Mumbai|            162301|           77282|            16|        1.8608|
|  Chennai|            124772|          135643|            16|        1.9963|
|Hyderabad|            113779|           90507|            13|        2.3602|
+---------+------------------+----------------+--------------+--------------+



#### Policy-type breakdown

In [0]:
policy_type_stats = policies_valid     .groupBy("policy_type")     .agg(
        count("policy_id").alias("policy_count"),
        sum("premium").alias("total_premium"),
        round(avg("premium"), 2).alias("avg_premium")
    )     .orderBy(col("total_premium").desc())

print("Policy type stats:")
policy_type_stats.show()


Policy type stats:
+-----------+------------+-------------+-----------+
|policy_type|policy_count|total_premium|avg_premium|
+-----------+------------+-------------+-----------+
|       Life|          26|       219825|    8454.81|
|     Health|          23|       202510|    8804.78|
|       Auto|          21|       154127|    7339.38|
+-----------+------------+-------------+-----------+



## Phase 5 – Advanced SQL using CTEs

#### Register temp views

In [0]:
customers_clean.createOrReplaceTempView("customers")
policies_valid.createOrReplaceTempView("policies")
claims_valid.createOrReplaceTempView("claims")
agents_clean.createOrReplaceTempView("agents")
policy_agent.createOrReplaceTempView("policy_agent")
customer_metrics.createOrReplaceTempView("customer_metrics")

print("Temp views registered.")


Temp views registered.


#### CTE — Top 3 risky customers per city

In [0]:
%sql
-- Step 1: aggregate premium per customer
-- Step 2: aggregate claims per customer
-- Step 3: compute risk score
-- Step 4: rank within each city
-- Step 5: filter top 3
WITH cust_premium AS (
    SELECT customer_id, SUM(premium) AS total_premium
    FROM policies
    GROUP BY customer_id
),
cust_claims AS (
    SELECT p.customer_id, SUM(c.claim_amount) AS total_claim
    FROM claims c
    JOIN policies p ON c.policy_id = p.policy_id
    GROUP BY p.customer_id
),
risk_scores AS (
    SELECT
        cu.customer_id,
        cu.name,
        cu.city,
        COALESCE(cp.total_premium, 0) AS total_premium,
        COALESCE(cc.total_claim, 0)   AS total_claim,
        CASE
            WHEN COALESCE(cp.total_premium, 0) = 0 THEN NULL
            ELSE ROUND(COALESCE(cc.total_claim, 0) / cp.total_premium, 4)
        END AS risk_score
    FROM customers cu
    LEFT JOIN cust_premium cp ON cu.customer_id = cp.customer_id
    LEFT JOIN cust_claims  cc ON cu.customer_id = cc.customer_id
),
ranked AS (
    SELECT *,
        DENSE_RANK() OVER (PARTITION BY city ORDER BY risk_score DESC NULLS LAST) AS city_rank
    FROM risk_scores
)
SELECT customer_id, name, city, total_premium, total_claim, risk_score, city_rank
FROM ranked
WHERE city_rank <= 3
ORDER BY city, city_rank


customer_id,name,city,total_premium,total_claim,risk_score,city_rank
58,Cust58,Bangalore,2618,10908,4.1665,1
32,Cust32,Bangalore,13457,24384,1.812,2
1,Cust1,Bangalore,17437,29747,1.706,3
54,Cust54,Chennai,2962,23605,7.9693,1
21,Cust21,Chennai,6560,19279,2.9389,2
2,Cust2,Chennai,872,2059,2.3612,3
46,Cust46,Hyderabad,378,7373,19.5053,1
5,Cust5,Hyderabad,5836,14832,2.5415,2
19,Cust19,Hyderabad,17219,36814,2.138,3
23,Cust23,Mumbai,1102,13117,11.9029,1


#### CTE — Customers with increasing claims month-over-month

In [0]:
%sql
WITH monthly_claims AS (
    SELECT
        p.customer_id,
        DATE_FORMAT(c.claim_date, 'yyyy-MM') AS claim_month,
        SUM(c.claim_amount)                  AS monthly_total
    FROM claims c
    JOIN policies p ON c.policy_id = p.policy_id
    GROUP BY p.customer_id, DATE_FORMAT(c.claim_date, 'yyyy-MM')
),
with_lag AS (
    SELECT
        customer_id,
        claim_month,
        monthly_total,
        LAG(monthly_total) OVER (PARTITION BY customer_id ORDER BY claim_month) AS prev_month_total
    FROM monthly_claims
),
increasing AS (
    SELECT customer_id, claim_month, monthly_total, prev_month_total
    FROM with_lag
    WHERE monthly_total > prev_month_total
)
SELECT i.customer_id, cu.name, cu.city, i.claim_month, i.monthly_total, i.prev_month_total,
       ROUND((i.monthly_total - i.prev_month_total) / i.prev_month_total * 100, 2) AS pct_increase
FROM increasing i
JOIN customers cu ON i.customer_id = cu.customer_id
ORDER BY pct_increase DESC


customer_id,name,city,claim_month,monthly_total,prev_month_total,pct_increase
17,Cust17,Chennai,2023-07,13267,1400,847.64
18,Cust18,Mumbai,2023-08,7369,1334,452.4
32,Cust32,Bangalore,2023-10,8969,2053,336.87
45,Cust45,Mumbai,2023-10,3817,927,311.76
19,Cust19,Hyderabad,2023-04,25612,9805,161.21
1,Cust1,Bangalore,2023-03,13997,5569,151.34
11,Cust11,Hyderabad,2023-07,5072,2476,104.85
4,Cust4,Hyderabad,2023-06,13644,10296,32.52
54,Cust54,Chennai,2023-09,13249,10356,27.94
21,Cust21,Chennai,2023-08,9574,8419,13.72


#### CTE — Monthly premium and claim trend

In [0]:
%sql
WITH monthly_premium AS (
    SELECT DATE_FORMAT(start_date, 'yyyy-MM') AS month, SUM(premium) AS total_premium
    FROM policies
    GROUP BY DATE_FORMAT(start_date, 'yyyy-MM')
),
monthly_claims AS (
    SELECT DATE_FORMAT(claim_date, 'yyyy-MM') AS month, SUM(claim_amount) AS total_claim
    FROM claims
    GROUP BY DATE_FORMAT(claim_date, 'yyyy-MM')
)
SELECT
    COALESCE(mp.month, mc.month) AS month,
    COALESCE(mp.total_premium, 0) AS total_premium,
    COALESCE(mc.total_claim, 0)   AS total_claim,
    ROUND(COALESCE(mc.total_claim, 0) / COALESCE(mp.total_premium, 1) * 100, 2) AS loss_ratio_pct
FROM monthly_premium mp
FULL OUTER JOIN monthly_claims mc ON mp.month = mc.month
ORDER BY month


month,total_premium,total_claim,loss_ratio_pct
2023-01,46520,46500,99.96
2023-02,86704,33244,38.34
2023-03,51265,76700,149.61
2023-04,16613,56656,341.03
2023-05,97172,23137,23.81
2023-06,27400,60595,221.15
2023-07,43373,41444,95.55
2023-08,86852,33908,39.04
2023-09,37298,42256,113.29
2023-10,83265,20470,24.58


## Phase 6 – Window Functions

#### Rank agents by total premium handled

In [0]:
agent_premium = policy_agent     .join(policies_valid, "policy_id")     .join(agents_clean.withColumnRenamed("name", "agent_name"), "agent_id")     .groupBy("agent_id", "agent_name", "region")     .agg(
        sum("premium").alias("total_premium_handled"),
        count("policy_id").alias("policies_handled")
    )

w_agent = Window.orderBy(col("total_premium_handled").desc())
w_region = Window.partitionBy("region").orderBy(col("total_premium_handled").desc())

agent_ranked = agent_premium     .withColumn("overall_rank",  rank().over(w_agent))     .withColumn("region_rank",   dense_rank().over(w_region))

print("Agent rankings (overall):")
agent_ranked.orderBy("overall_rank").display(20)


Agent rankings (overall):


/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


agent_id,agent_name,region,total_premium_handled,policies_handled,overall_rank,region_rank
14,Agent14,East,39883,4,1,1
10,Agent10,North,37613,3,2,1
20,Agent20,North,37593,6,3,2
13,Agent13,South,36692,4,4,1
11,Agent11,West,32466,3,5,1
7,Agent7,South,30257,2,6,2
1,Agent1,South,27707,5,7,3
8,Agent8,North,27553,2,8,3
5,Agent5,South,27343,7,9,4
15,Agent15,North,27201,5,10,4


#### Top agents per region using DENSE_RANK

In [0]:
top_agents_per_region = agent_ranked.filter(col("region_rank") <= 3)     .orderBy("region", "region_rank")

print("Top 3 agents per region:")
top_agents_per_region.display()


Top 3 agents per region:


/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


agent_id,agent_name,region,total_premium_handled,policies_handled,overall_rank,region_rank
14,Agent14,East,39883,4,1,1
9,Agent9,East,25259,4,13,2
19,Agent19,East,25176,4,14,3
10,Agent10,North,37613,3,2,1
20,Agent20,North,37593,6,3,2
8,Agent8,North,27553,2,8,3
13,Agent13,South,36692,4,4,1
7,Agent7,South,30257,2,6,2
1,Agent1,South,27707,5,7,3
11,Agent11,West,32466,3,5,1


#### Rank customers by risk score within each city

In [0]:
w_city = Window.partitionBy("city").orderBy(col("risk_score").desc_nulls_last())

customer_risk_ranked = customer_metrics     .withColumn("city_risk_rank", dense_rank().over(w_city))     .select("customer_id", "name", "city", "total_premium",
            "total_claim", "risk_score", "city_risk_rank")     .orderBy("city", "city_risk_rank")

print("Customer risk ranking per city:")
customer_risk_ranked.display(20)

Customer risk ranking per city:


customer_id,name,city,total_premium,total_claim,risk_score,city_risk_rank
58,Cust58,Bangalore,2618,10908,4.1665,1
32,Cust32,Bangalore,13457,24384,1.812,2
1,Cust1,Bangalore,17437,29747,1.706,3
16,Cust16,Bangalore,14656,24500,1.6717,4
47,Cust47,Bangalore,16608,26647,1.6045,5
12,Cust12,Bangalore,25754,6667,0.2589,6
28,Cust28,Bangalore,31933,5878,0.1841,7
50,Cust50,Bangalore,21242,2747,0.1293,8
3,Cust3,Bangalore,4360,0,0.0,9
25,Cust25,Bangalore,11188,0,0.0,9


#### Running total premium per customer (ordered by policy start date)

In [0]:
w_running = Window.partitionBy("customer_id").orderBy("start_date")     .rowsBetween(Window.unboundedPreceding, Window.currentRow)

running_premium = policies_valid     .withColumn("running_premium", sum("premium").over(w_running))     .select("customer_id", "policy_id", "policy_type", "start_date", "premium", "running_premium")     .orderBy("customer_id", "start_date")

print("Running premium per customer:")
running_premium.display(15)


Running premium per customer:


customer_id,policy_id,policy_type,start_date,premium,running_premium
1,174,Auto,2023-03-12,10181,10181
1,170,Life,2023-05-22,7256,17437
2,163,Auto,2023-05-05,872,872
3,132,Health,2023-07-28,4360,4360
4,127,Health,2023-07-08,1795,1795
4,166,Health,2023-07-19,14924,16719
5,141,Auto,2023-04-28,1909,1909
5,124,Life,2023-05-12,3927,5836
6,164,Health,2023-01-29,3629,3629
8,150,Health,2023-06-24,9978,9978


## Phase 5 (SQL) – Agent Analytics

#### Agent region contribution

In [0]:
agent_ranked.createOrReplaceTempView("agent_ranked")


/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
%sql
WITH region_totals AS (
    SELECT region, SUM(total_premium_handled) AS region_total
    FROM agent_ranked
    GROUP BY region
)
SELECT
    a.agent_id,
    a.agent_name,
    a.region,
    a.total_premium_handled,
    a.policies_handled,
    rt.region_total,
    ROUND(a.total_premium_handled / rt.region_total * 100, 2) AS region_contribution_pct,
    a.region_rank
FROM agent_ranked a
JOIN region_totals rt ON a.region = rt.region
ORDER BY a.region, a.region_rank


agent_id,agent_name,region,total_premium_handled,policies_handled,region_total,region_contribution_pct,region_rank
14,Agent14,East,39883,4,103376,38.58,1
9,Agent9,East,25259,4,103376,24.43,2
19,Agent19,East,25176,4,103376,24.35,3
6,Agent6,East,13058,5,103376,12.63,4
10,Agent10,North,37613,3,170849,22.02,1
20,Agent20,North,37593,6,170849,22.0,2
8,Agent8,North,27553,2,170849,16.13,3
15,Agent15,North,27201,5,170849,15.92,4
2,Agent2,North,26185,3,170849,15.33,5
16,Agent16,North,14704,1,170849,8.61,6


## Phase 7 – Final Output & Validation

#### Global validation checks

In [0]:
print("=" * 60)
print("GLOBAL VALIDATION CHECKS")
print("=" * 60)

# 1. No negative premiums in final dataset
neg_premium = policies_valid.filter(col("premium") < 0).count()
print(f"  Negative premiums in final dataset      : {neg_premium} (expect 0)")

# 2. No negative claim amounts in final dataset
neg_claims = claims_valid.filter(col("claim_amount") < 0).count()
print(f"  Negative claim_amounts in final dataset : {neg_claims} (expect 0)")

# 3. No nulls in reporting columns
nulls_cm = customer_metrics.filter(
    col("customer_id").isNull() | col("city").isNull()
).count()
print(f"  Nulls in customer_metrics key columns   : {nulls_cm} (expect 0)")

# 4. Total premium consistency
total_p_policies = policies_valid.agg(sum("premium")).collect()[0][0]
total_p_metrics  = customer_metrics.agg(sum("total_premium")).collect()[0][0]
print(f"  Total premium (policies)  : {total_p_policies}")
print(f"  Total premium (metrics)   : {total_p_metrics}")
print(f"customer_metrics count: {customer_metrics.count()}")
print(f"customers_clean count: {customers_clean.count()}")
print(f"Match: {customer_metrics.count() == customers_clean.count()}")

# 5. Row count did not explode in join
print(f"  customer_metrics rows     : {customer_metrics.count()} (expect = {customers_clean.count()})")

print("=" * 60)


GLOBAL VALIDATION CHECKS
  Negative premiums in final dataset      : 0 (expect 0)
  Negative claim_amounts in final dataset : 0 (expect 0)
  Nulls in customer_metrics key columns   : 0 (expect 0)
  Total premium (policies)  : 576462
  Total premium (metrics)   : 576462
customer_metrics count: 60
customers_clean count: 60
Match: True
  customer_metrics rows     : 60 (expect = 60)


#### Save all final outputs

In [0]:
OUT = "/Volumes/workspace/default/week2-day1/output"

customer_metrics.write.mode("overwrite").option("header", True).csv(f"{OUT}/customer_metrics")
city_distribution.write.mode("overwrite").option("header", True).csv(f"{OUT}/city_distribution")
policy_type_stats.write.mode("overwrite").option("header", True).csv(f"{OUT}/policy_type_stats")
agent_ranked.write.mode("overwrite").option("header", True).csv(f"{OUT}/agent_ranked")
customer_risk_ranked.write.mode("overwrite").option("header", True).csv(f"{OUT}/customer_risk_ranked")

print("All outputs saved to:", OUT)


/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


All outputs saved to: /Volumes/workspace/default/week2-day1/output
